# NSMC 한국어 감성 분류

네이버 영화 리뷰 데이터를 허깅페이스에서 가져와서 리뷰가 긍정인지 부정인지 감정하는 AI 만들기

In [12]:
!pip install evaluate datasets transformers scikit-learn accelerate

import numpy as np
import evaluate
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"사용 중인 장치: {device}")

사용 중인 장치: cuda


## 1. 데이터 가져오기

15만개는 train  / 5만개는 test로 넣기

In [2]:
dataset = load_dataset("csv", data_files={
    "train": "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt",
    "test": "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt"
}, delimiter="\t")

# 데이터 정제 (빈 글자 제외)
dataset = dataset.filter(lambda x: x['document'] is not None and len(str(x['document'])) > 0)

train_dataset = dataset["train"].shuffle(seed=42)
test_dataset = dataset["test"].shuffle(seed=42)

print(f"{len(train_dataset)}개")

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/150000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/50000 [00:00<?, ? examples/s]

149995개


## 2. klue/bert-base model 및 tokenizer 불러오기

klue/bert-base 한국어 사전학습된 BERT모델 \
num_labels=2: 긍정(1) / 부정(0) 분류

In [13]:
model_name = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 3. tokenizer으로 데이터셋을 전처리하고, model 학습

모든 문장 padding 'max length'로 128 길이로 통일

In [8]:
def preprocess_function(examples):
    texts = [str(doc) for doc in examples['document']]
    return tokenizer(texts, truncation=True, padding='max_length', max_length=128)

tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_test = test_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/149995 [00:00<?, ? examples/s]

Map:   0%|          | 0/49997 [00:00<?, ? examples/s]

## 4. Fine-tuning을 통하여 모델 성능(accuarcy) 향상

batch size 32\
epoch 3번으로 돌려보기

In [6]:
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="./gpu_movie_results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32, 
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=True, # GPU 가속 활성화
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.249826,0.236237,0.903414
2,0.175195,0.262515,0.906654
3,0.109552,0.306612,0.906934


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=14064, training_loss=0.19065660936704945, metrics={'train_runtime': 3358.5684, 'train_samples_per_second': 133.981, 'train_steps_per_second': 4.187, 'total_flos': 2.95990070615424e+16, 'train_loss': 0.19065660936704945, 'epoch': 3.0})

In [9]:
# 최종 평가 진행하기
results = trainer.evaluate()

# 최종 정확도 확인하기
print(f"{results['eval_accuracy'] * 100:.2f}%")

Training Loss,Validation Loss,Epoch,Accuracy
0.109552,0.306612,3,0.906934


90.69%


이미 정확도 90으로 높게 나옴.

In [12]:
def simple_predict(text):
    # 문장을 숫자로 변환
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding='max_length', max_length=128).to(device)
    # 모델에게 질문
    outputs = model(**inputs)
    # 0(부정) 혹은 1(긍정) 중 큰 값 찾기
    prediction = np.argmax(outputs.logits.detach().cpu().numpy(), axis=-1)
    
    return "긍정 " if prediction == 1 else "부정 "


In [13]:
# 결과 확인
print(simple_predict("영화 한번쯤 볼 만하네."))

부정 


In [39]:
import torch.nn.functional as F

def predict_sentiment_pro(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding='max_length', max_length=128).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    
    # AI가 계산한 각 확률
    probs = F.softmax(outputs.logits, dim=-1).detach().cpu().numpy()[0]
    
    neg_prob = probs[0]
    pos_prob = probs[1]

    if pos_prob > neg_prob:
        return f"긍정 ({pos_prob*100:.2f}%) "
    else:
        return f"부정 ({neg_prob*100:.2f}%) "

# 테스트용 문장
test_sentences = [
    "대박. 개재밌어요",
    "돈 아깝지않은 내용!",
    "쏘쏘",
    "보통이에요",
    "돈 아까워요. 내용도 없고 지루해요. 돈낭비네요.",
    "그냥 그랬어요. 그저 그랬어요.",
    "배우들 연기는 좋은데 스토리가 좀 아쉽네요.",
    "영화 앞부분은 좋고 뒷부분은 별로에요.",
    "영화 앞부분이 좋고 나머지는 보통 정도에요.",
    "영화 앞부분이 좋아요."
]


print("--- 감정 결과 ---")
for sentence in test_sentences:
    print(f"문장: {sentence}")
    print(f"판정: {predict_sentiment_pro(sentence)}\n")

--- 감정 결과 ---
문장: 대박. 개재밌어요
판정: 긍정 (96.60%) 

문장: 돈 아깝지않은 내용!
판정: 긍정 (99.21%) 

문장: 쏘쏘
판정: 긍정 (94.11%) 

문장: 보통이에요
판정: 부정 (99.36%) 

문장: 돈 아까워요. 내용도 없고 지루해요. 돈낭비네요.
판정: 부정 (99.96%) 

문장: 그냥 그랬어요. 그저 그랬어요.
판정: 부정 (99.73%) 

문장: 배우들 연기는 좋은데 스토리가 좀 아쉽네요.
판정: 부정 (94.45%) 

문장: 영화 앞부분은 좋고 뒷부분은 별로에요.
판정: 부정 (99.18%) 

문장: 영화 앞부분이 좋고 나머지는 보통 정도에요.
판정: 부정 (99.58%) 

문장: 영화 앞부분이 좋아요.
판정: 긍정 (53.58%) 



문장 테스트하면 다 잘 나옴. 긍정,부정만 있어서 쏘쏘나 보통같은 중간값도 그냥 부정으로 확 쏠리는 것 같음\
이건 중립 값을 앞에 넣어주면 될 것 같음.

## 5. Bucketing을 적용하여 학습시키고, STEP 4의 결과와의 비교

기존에 모든 문장 128로 고정해서 짧은 문장도 패딩을 다 넣게 되는데,\
버켓팅으로 비슷한 길이 문장끼리 묶어서 배치해서 불필요한 패딩 줄이기

[변경사항]
* 전처리에서 padding 제거 → 문장마다 실제 길이 보존\
* train_sampling_strategy="group_by_length" → 비슷한 길이끼리 묶기\
* DataCollatorWithPadding → 배치마다 최소 패딩 자동 적용

In [16]:
from transformers import DataCollatorWithPadding

In [17]:
def preprocess_function_bucketing(examples):
    texts = [str(doc) for doc in examples['document']]
    return tokenizer(
        texts,
        truncation=True,   
        max_length=128,    
    )
# 전처리 다시 실행 
tokenized_train_b = train_dataset.map(preprocess_function_bucketing, batched=True)
tokenized_test_b  = test_dataset.map(preprocess_function_bucketing,  batched=True)

print("전처리 완료!")
print(f"  예시 문장 실제 토큰 길이: {len(tokenized_train_b[0]['input_ids'])}") 


Map:   0%|          | 0/149995 [00:00<?, ? examples/s]

Map:   0%|          | 0/49997 [00:00<?, ? examples/s]

전처리 완료!
  예시 문장 실제 토큰 길이: 36


36으로 128이 아닌 실제 길이로 나옴

In [18]:
training_args_bucketing = TrainingArguments(
    output_dir="./bucketing_results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=True,
    train_sampling_strategy="group_by_length",  # 비슷한 길이끼리 배치로 묶음
)

In [19]:

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8
)

# 이전 학습 가중치 영향 없이 다시 학습
model_bucketing = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=2
).to(device)

# training_args_bucketing, tokenized_train_b/test_b, data_collator 사용
trainer_bucketing = Trainer(
    model=model_bucketing,
    args=training_args_bucketing,
    train_dataset=tokenized_train_b,   
    eval_dataset=tokenized_test_b,     
    compute_metrics=compute_metrics,   
    data_collator=data_collator,       
)

trainer_bucketing.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.255845,0.246308,0.902754
2,0.178230,0.243885,0.906754
3,0.116265,0.307983,0.906974


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=14064, training_loss=0.19196052186453844, metrics={'train_runtime': 1763.9261, 'train_samples_per_second': 255.104, 'train_steps_per_second': 7.973, 'total_flos': 6251654341991520.0, 'train_loss': 0.19196052186453844, 'epoch': 3.0})

In [21]:
results_bucketing = trainer_bucketing.evaluate()
print(f"버켓팅 정확도: {results_bucketing['eval_accuracy'] * 100:.2f}%")

Training Loss,Validation Loss,Epoch,Accuracy
0.116265,0.307985,3,0.906994


버켓팅 정확도: 90.70%


In [22]:
import torch.nn.functional as F

def predict_sentiment_bucketing(text):
    inputs = tokenizer(
        text, 
        return_tensors="pt", 
        truncation=True, 
        padding=True,       
        max_length=128
    ).to(device)
    
    with torch.no_grad():
        outputs = model_bucketing(**inputs)  
    probs = F.softmax(outputs.logits, dim=-1).detach().cpu().numpy()[0]
    
    neg_prob = probs[0]
    pos_prob = probs[1]
    
    if pos_prob > neg_prob:
        return f"긍정 ({pos_prob*100:.2f}%)"
    else:
        return f"부정 ({neg_prob*100:.2f}%)"

# 테스트 문장 (이전과 동일)
test_sentences = [
    "대박. 개재밌어요",
    "돈 아깝지않은 내용!",
    "쏘쏘",
    "보통이에요",
    "돈 아까워요. 내용도 없고 지루해요. 돈낭비네요.",
    "그냥 그랬어요. 그저 그랬어요.",
    "배우들 연기는 좋은데 스토리가 좀 아쉽네요.",
    "영화 앞부분은 좋고 뒷부분은 별로에요.",
    "영화 앞부분이 좋고 나머지는 보통 정도에요.",
    "영화 앞부분이 좋아요."
]

print("--- 버켓팅 모델 감정 결과 ---")
for sentence in test_sentences:
    print(f"문장: {sentence}")
    print(f"판정: {predict_sentiment_bucketing(sentence)}\n")

--- 버켓팅 모델 감정 결과 ---
문장: 대박. 개재밌어요
판정: 긍정 (96.53%)

문장: 돈 아깝지않은 내용!
판정: 긍정 (99.82%)

문장: 쏘쏘
판정: 긍정 (93.26%)

문장: 보통이에요
판정: 부정 (96.19%)

문장: 돈 아까워요. 내용도 없고 지루해요. 돈낭비네요.
판정: 부정 (99.97%)

문장: 그냥 그랬어요. 그저 그랬어요.
판정: 부정 (99.70%)

문장: 배우들 연기는 좋은데 스토리가 좀 아쉽네요.
판정: 부정 (96.81%)

문장: 영화 앞부분은 좋고 뒷부분은 별로에요.
판정: 부정 (98.99%)

문장: 영화 앞부분이 좋고 나머지는 보통 정도에요.
판정: 부정 (99.56%)

문장: 영화 앞부분이 좋아요.
판정: 부정 (57.87%)



#### Trade-off 분석 — 기존 vs 버켓팅 비교

In [24]:
# 기존 학습 수치
original_runtime  = 3358.5684
original_accuracy = 90.69    # Epoch3 기준

# 버켓팅 학습 수치
bucketing_runtime  = 1763.9261
bucketing_accuracy = 90.70   # Epoch3 기준

# 계산 
time_saved     = original_runtime - bucketing_runtime
time_saved_pct = (time_saved / original_runtime) * 100
acc_diff       = bucketing_accuracy - original_accuracy
speed_ratio    = original_runtime / bucketing_runtime  # 몇 배 빠른지


In [27]:
print(f"\n{'항목':<10} {'기존 방식':>10} {'버켓팅':>10}")
print("-" * 55)
print(f"{'학습 시간':<10} {original_runtime:>12.1f}초  {bucketing_runtime:>12.1f}초")
print(f"{'학습 시간(분)':<10} {original_runtime/60:>12.1f}분  {bucketing_runtime/60:>12.1f}분")
print(f"{'정확도':<10} {original_accuracy:>13.2f}%  {bucketing_accuracy:>13.2f}%")

print(f"\n{'[ 비교 결과 ]':=<55}")
print(f"  절약된 시간  : {time_saved:.1f}초 ({time_saved/60:.1f}분)")
print(f"  속도 향상    : {speed_ratio:.2f}배 빠름 ({time_saved_pct:.1f}% 단축)")
print(f"  정확도 변화  : {acc_diff:+.2f}%")

print(f"\n{'[ 결론 ]':=<55}")
print(f"  정확도는 거의 동일 ({acc_diff:+.2f}%)")
print(f"  학습 시간은 {speed_ratio:.1f}배 단축")
print(f"  → Bucketing은 성능 손실 없이 속도만 개선!")


항목              기존 방식        버켓팅
-------------------------------------------------------
학습 시간            3358.6초        1763.9초
학습 시간(분)           56.0분          29.4분
정확도                90.69%          90.70%

[ 비교 결과 ]==============================================
  절약된 시간  : 1594.6초 (26.6분)
  속도 향상    : 1.90배 빠름 (47.5% 단축)
  정확도 변화  : +0.01%

[ 결론 ]=================================================
  정확도는 거의 동일 (+0.01%)
  학습 시간은 1.9배 단축
  → Bucketing은 성능 손실 없이 속도만 개선!


In [28]:
print(f"{'Epoch':<8} {'기존 Acc':>10} {'버켓팅 Acc':>12} {'기존 ValLoss':>14} {'버켓팅 ValLoss':>15}")

original_epochs = [
    (1, 0.903414, 0.236237),
    (2, 0.906654, 0.262515),
    (3, 0.906934, 0.306612),
]
bucketing_epochs = [
    (1, 0.902754, 0.246308),
    (2, 0.906754, 0.243885),
    (3, 0.906974, 0.307983),
]

for (ep, orig_acc, orig_loss), (_, buck_acc, buck_loss) in zip(original_epochs, bucketing_epochs):
    diff = (buck_acc - orig_acc) * 100
    print(f"  {ep:<6} {orig_acc*100:>9.2f}%  {buck_acc*100:>11.2f}%  "
          f"{orig_loss:>13.6f}  {buck_loss:>14.6f}  ({diff:+.2f}%)")

print("-" * 65)
print(f"\n최적 에포크: Epoch 2")
print(f"   기존    — Acc: 90.67%, Val Loss: 0.2625")
print(f"   버켓팅  — Acc: 90.68%, Val Loss: 0.2439 ← Val Loss 더 낮음!")
print(f"   → Epoch 2에서는 버켓팅이 정확도 + Val Loss 모두 우세!")

Epoch        기존 Acc      버켓팅 Acc     기존 ValLoss     버켓팅 ValLoss
  1          90.34%        90.28%       0.236237        0.246308  (-0.07%)
  2          90.67%        90.68%       0.262515        0.243885  (+0.01%)
  3          90.69%        90.70%       0.306612        0.307983  (+0.00%)
-----------------------------------------------------------------

최적 에포크: Epoch 2
   기존    — Acc: 90.67%, Val Loss: 0.2625
   버켓팅  — Acc: 90.68%, Val Loss: 0.2439 ← Val Loss 더 낮음!
   → Epoch 2에서는 버켓팅이 정확도 + Val Loss 모두 우세!


#### 감정 분석 결과 비교 - 기존 모델 vs 버켓팅 모델

In [36]:

import torch.nn.functional as F

# 기존 모델 추론 함수
def predict_original(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding='max_length',  # 기존 방식 그대로
        max_length=128
    ).to(device)
    with torch.no_grad():
        outputs = model(**inputs)          # 기존 model 사용
    probs = F.softmax(outputs.logits, dim=-1).detach().cpu().numpy()[0]
    neg_prob, pos_prob = probs[0], probs[1]
    label = "긍정" if pos_prob > neg_prob else "부정"
    prob  = max(pos_prob, neg_prob) * 100
    return label, prob

# 버켓팅 모델 추론 함수
def predict_bucketing(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,          # 동적 패딩
        max_length=128
    ).to(device)
    with torch.no_grad():
        outputs = model_bucketing(**inputs)  # 버켓팅 model 사용
    probs = F.softmax(outputs.logits, dim=-1).detach().cpu().numpy()[0]
    neg_prob, pos_prob = probs[0], probs[1]
    label = "긍정" if pos_prob > neg_prob else "부정"
    prob  = max(pos_prob, neg_prob) * 100
    return label, prob

# 테스트 문장
test_sentences = [
    "대박. 개재밌어요",
    "돈 아깝지않은 내용!",
    "쏘쏘",
    "보통이에요",
    "돈 아까워요. 내용도 없고 지루해요. 돈낭비네요.",
    "그냥 그랬어요. 그저 그랬어요.",
    "배우들 연기는 좋은데 스토리가 좀 아쉽네요.",
    "영화 앞부분은 좋고 뒷부분은 별로에요.",
    "영화 앞부분이 좋고 나머지는 보통 정도에요.",
    "영화 앞부분이 좋아요."
]

# 비교 출력
print("=" * 75)
print(f"{'문장':<25} {'기존모델':>15} {'버켓팅모델':>15} {'일치':>8}")
print("=" * 75)

agree = 0
for sentence in test_sentences:
    orig_label, orig_prob = predict_original(sentence)
    buck_label, buck_prob = predict_bucketing(sentence)

    match = "O" if orig_label == buck_label else "X 다름"
    if orig_label == buck_label:
        agree += 1

    display = sentence[:22] + ".." if len(sentence) > 22 else sentence

    print(f"{display:<25} "
          f"{orig_label}({orig_prob:5.1f}%)   "
          f"{buck_label}({buck_prob:5.1f}%)   "
          f"{match}")

print("=" * 75)
print(f"\n결과 요약")
print(f"  전체 문장   : {len(test_sentences)}개")
print(f"  판정 일치   : {agree}개")
print(f"  판정 불일치 : {len(test_sentences) - agree}개")
print(f"  일치율      : {agree/len(test_sentences)*100:.0f}%")

문장                                   기존모델           버켓팅모델       일치
대박. 개재밌어요                 긍정( 50.6%)   긍정( 96.5%)   O
돈 아깝지않은 내용!               긍정( 60.0%)   긍정( 99.8%)   O
쏘쏘                        긍정( 56.3%)   긍정( 93.3%)   O
보통이에요                     긍정( 59.8%)   부정( 96.2%)   X 다름
돈 아까워요. 내용도 없고 지루해요. 돈..  긍정( 53.4%)   부정(100.0%)   X 다름
그냥 그랬어요. 그저 그랬어요.         긍정( 56.5%)   부정( 99.7%)   X 다름
배우들 연기는 좋은데 스토리가 좀 아쉽네..  부정( 50.5%)   부정( 96.8%)   O
영화 앞부분은 좋고 뒷부분은 별로에요.     부정( 51.1%)   부정( 99.0%)   O
영화 앞부분이 좋고 나머지는 보통 정도에..  부정( 52.0%)   부정( 99.6%)   O
영화 앞부분이 좋아요.              부정( 52.6%)   부정( 57.9%)   O

결과 요약
  전체 문장   : 10개
  판정 일치   : 7개
  판정 불일치 : 3개
  일치율      : 70%


# 회고


이미 데이터가 잘되어있어서 그냥 돌려도 90퍼로 나왔다. 하지만 버켓팅을 했을때 학습시간이 반정도 줄었고 accuracy는 거의 비슷했다.\
기존방식으로 하면 모든 문장 같은 길이로 강제로 통일 시켜서 짧은 문장도 패딩을 넣어서 계산속도가 느림\
-> 그러니 버켓팅으로 쓰자..

트랜스포머 버전에 따라서 파라미터가 달라서
처음에 group_by_length=True 가 최신 버전에서 TypeError가 발생했고,
train_sampling_strategy="group_by_length" 로 변경했다.

그리고 계속 수정하고 돌리다보니까 중간에는 디스크가 꽉차서 다시 커널을 돌리고 전에 한거를 지우고 돌리기도 했다.

학습하면서 에폭3에서 loss가 올라가면서 과적합 되려고 하는데 여기에는 earlystopping을 하면 좋을 것 같다.

다른 데이터를 하고 싶었으나 찾지 못해서 우선 네이버 데이터로 했는데 확실히 잘 정리되어 있어서 그냥 돌려도 굉장히 높게 나와서 좋았다.\
다음에 다른 데이터로 좀 여러가시 실험해 봐야 될 것 같다.
